### **Cell 1: Khai báo thư viện & Kiểm tra môi trường**
- Import các thư viện cốt lõi (`TensorFlow`, `OpenCV`, `NumPy`, `Matplotlib`).
- Kiểm tra phiên bản TensorFlow và thiết bị GPU khả dụng.


In [1]:
import os
import glob
import ast
import numpy as np
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf


print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices("GPU"))


TensorFlow Version: 2.21.0
GPU Available: []


### **Cell 2: Thiết lập đường dẫn dữ liệu & Siêu tham số**
- Khai báo đường dẫn chứa ảnh gốc, file nhãn txt, và thư mục lưu mask/model.
- Cấu hình kích thước ảnh đầu vào ($256 \times 256$), Batch Size ($16$).


In [2]:
DATA_DIR = r"e:\HOCKY7\PBL6\code_final\ai-service\data\1_segmentation"
IMG_DIR = os.path.join(DATA_DIR, "imgs")
TXT_DIR = os.path.join(DATA_DIR, "txt")
MASK_DIR = os.path.join(DATA_DIR, "masks")
MODEL_SAVE_PATH = r"e:\HOCKY7\PBL6\code_final\ai-service\models\1_segmentation\receipt_unet_mobilenetv2.keras"

# 2. Siêu tham số huấn luyện
IMG_HEIGHT = 256  # Chiều cao chuẩn hóa
IMG_WIDTH = 256   # Chiều rộng chuẩn hóa
BATCH_SIZE = 16   # Kích thước mỗi batch
BUFFER_SIZE = 1000


### **Cell 3: Tạo mặt nạ nhị phân (Binary Mask) từ tọa độ đa giác**
- Đọc tọa độ polygon hóa đơn từ file `.txt`.
- Vẽ đa giác màu trắng ($255$) trên nền đen ($0$) và lưu thành ảnh `.png` trong thư mục `masks/`.


In [ ]:
def create_masks_from_annotations():
    """Tạo file mask nhị phân từ file tọa độ txt."""
    os.makedirs(MASK_DIR, exist_ok=True)
    img_files = [f for f in os.listdir(IMG_DIR) if f.endswith(".jpg")]
    
    count = 0
    for img_file in img_files:
        base_name = os.path.splitext(img_file)[0]
        txt_path = os.path.join(TXT_DIR, base_name + ".txt")
        mask_path = os.path.join(MASK_DIR, base_name + ".png")
        
        # Bỏ qua nếu mask đã có hoặc thiếu file txt
        if os.path.exists(mask_path) or not os.path.exists(txt_path):
            continue
            
        img = cv2.imread(os.path.join(IMG_DIR, img_file))
        if img is None:
            continue
            
        h, w = img.shape[:2]
        mask = np.zeros((h, w), dtype=np.uint8)
        
        with open(txt_path, "r", encoding="utf-8") as f:
            content = f.read().strip()
            
        try:
            pts = np.array(ast.literal_eval(f"[{content}]"), dtype=np.int32)
            cv2.fillPoly(mask, [pts], 255)  # Vẽ vùng hóa đơn màu trắng (255)
            cv2.imwrite(mask_path, mask)
            count += 1
        except Exception as e:
            print(f"Lỗi file {txt_path}: {e}")
            
    print(f"Đã hoàn thành tạo mask! (Tạo mới: {count})")

# Thực thi tạo mask
create_masks_from_annotations()


### **Cell 4: Trực quan hóa & Lưu ảnh có viền Bounding (Kiểm tra nhãn)**
- Vẽ đường viền đỏ xung quanh hóa đơn theo tọa độ nhãn.
- Xuất ảnh vào `bounded_imgs/` để kiểm tra độ chính xác của tập dữ liệu.


In [ ]:
# Đường dẫn thư mục lưu ảnh đã vẽ bounding
OUTPUT_DIR = os.path.join(DATA_DIR, "bounded_imgs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

img_files = sorted(glob.glob(os.path.join(IMG_DIR, "*.jpg")))
print(f"Bắt đầu xử lý {len(img_files)} ảnh...")

count = 0
for img_path in img_files:
    base_name = os.path.splitext(os.path.basename(img_path))[0]
    txt_path = os.path.join(TXT_DIR, base_name + ".txt")
    out_path = os.path.join(OUTPUT_DIR, base_name + ".jpg")
    
    if not os.path.exists(txt_path):
        continue
        
    img = cv2.imread(img_path)
    if img is None:
        continue
        
    with open(txt_path, "r", encoding="utf-8") as f:
        content = f.read().strip()
        
    try:
        if "[" in content:
            pts = np.array(ast.literal_eval(f"[{content}]"), dtype=np.float32).reshape(-1, 2).astype(np.int32)
        else:
            pts = np.array([float(x) for x in content.split(",") if x.strip()], dtype=np.float32).reshape(-1, 2).astype(np.int32)
            
        # Vẽ viền đỏ dày 4px bao quanh hóa đơn
        cv2.polylines(img, [pts], isClosed=True, color=(0, 0, 255), thickness=4)
        cv2.imwrite(out_path, img)
        count += 1
    except Exception as e:
        print(f"Lỗi xử lý file {base_name}: {e}")

print(f"Hoàn tất! Đã xuất {count} ảnh có bounding sang: {OUTPUT_DIR}")


### **Cell 5: Thu thập danh sách các cặp (Ảnh, Mask) hợp lệ**
- Lọc và ghép cặp từng file ảnh `.jpg` với file mask `.png` tương ứng.
- Đếm tổng số lượng mẫu hợp lệ sẵn sàng cho huấn luyện.


In [ ]:
image_paths = []
mask_paths = []

for img_path in sorted(glob.glob(os.path.join(IMG_DIR, "*.jpg"))):
    base_name = os.path.splitext(os.path.basename(img_path))[0]
    m_path = os.path.join(MASK_DIR, base_name + ".png")
    
    # Chỉ lấy các ảnh có file mask tương ứng
    if os.path.exists(m_path):
        image_paths.append(img_path)
        mask_paths.append(m_path)

total_samples = len(image_paths)
print(f"Tổng số mẫu hợp lệ: {total_samples}")


### **Cell 6: Hàm nạp & Tăng cường dữ liệu (Data Pipeline & Augmentation)**
- `load_image_train`: Đọc ảnh/mask, resize $256 \times 256$, chuẩn hóa $[0, 1]$, lật ngang, đổi độ sáng & tương phản ngẫu nhiên.
- `load_image_test`: Đọc và chuẩn hóa ảnh/mask cho tập Validation.


In [ ]:
def load_image_train(image_path, mask_path):
    """Hàm nạp và tăng cường dữ liệu cho tập Train."""
    # Đọc và decode ảnh
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, (IMG_HEIGHT, IMG_WIDTH))
    image = tf.cast(image, tf.float32) / 255.0

    # Đọc và decode mask
    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_png(mask, channels=1)
    mask = tf.image.resize(mask, (IMG_HEIGHT, IMG_WIDTH), method=tf.image.ResizeMethod.NEAREST_NEIGHBOR)
    mask = tf.cast(mask, tf.float32) / 255.0

    # --- DATA AUGMENTATION ---
    # 1. Lật ngang ngẫu nhiên cả ảnh và mask
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_left_right(image)
        mask = tf.image.flip_left_right(mask)

    # 2. Thay đổi độ sáng ngẫu nhiên
    image = tf.image.random_brightness(image, max_delta=0.15)

    # 3. Thay đổi độ tương phản ngẫu nhiên
    image = tf.image.random_contrast(image, lower=0.85, upper=1.15)

    # Giữ giá trị pixel trong khoảng [0, 1]
    image = tf.clip_by_value(image, 0.0, 1.0)

    return image, mask

def load_image_test(image_path, mask_path):
    """Hàm nạp dữ liệu cho tập Validation (không augment)."""
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, (IMG_HEIGHT, IMG_WIDTH))
    image = tf.cast(image, tf.float32) / 255.0

    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_png(mask, channels=1)
    mask = tf.image.resize(mask, (IMG_HEIGHT, IMG_WIDTH), method=tf.image.ResizeMethod.NEAREST_NEIGHBOR)
    mask = tf.cast(mask, tf.float32) / 255.0

    return image, mask


### **Cell 7: Xây dựng DataLoader với `tf.data` (Train / Val Split)**
- Chia dữ liệu theo tỷ lệ **80% Train / 20% Validation**.
- Tối ưu pipeline với `shuffle()`, `batch()`, và `prefetch(AUTOTUNE)`.


In [ ]:
# Khởi tạo dataset từ mảng đường dẫn
dataset = tf.data.Dataset.from_tensor_slices((image_paths, mask_paths))

# Tách Train / Validation (80% / 20%)
train_size = int(0.8 * total_samples)
train_raw = dataset.take(train_size)
val_raw = dataset.skip(train_size)

# DataLoader tối ưu cho Train và Validation
train_dataset = (
    train_raw
    .map(load_image_train, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE)
    .prefetch(buffer_size=tf.data.AUTOTUNE)
)

val_dataset = (
    val_raw
    .map(load_image_test, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(buffer_size=tf.data.AUTOTUNE)
)

print(f"Số lượng Train batches: {len(train_dataset)}")
print(f"Số lượng Val batches: {len(val_dataset)}")


### **Cell 8: Trực quan hóa tỷ lệ phân chia Dataset**
- Vẽ biểu đồ cột (Bar Chart) và biểu đồ tròn (Pie Chart) thể hiện số lượng và tỷ lệ mẫu giữa tập Train và Validation.


In [ ]:
val_size = total_samples - train_size
splits = ["Train Set", "Validation Set"]
counts = [train_size, val_size]
colors = ["#3498db", "#e74c3c"]

plt.figure(figsize=(12, 5))

# 1. Biểu đồ cột thể hiện số lượng mẫu
plt.subplot(1, 2, 1)
bars = plt.bar(splits, counts, color=colors, width=0.5, edgecolor="black", alpha=0.85)
plt.title("Dataset Split (Sample Count)", fontsize=13, fontweight="bold")
plt.ylabel("Số lượng ảnh", fontsize=11)
plt.grid(axis="y", linestyle=":", alpha=0.7)
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 15, f"{int(yval)}", ha="center", va="bottom", fontsize=11, fontweight="bold")

# 2. Biểu đồ tròn thể hiện tỷ lệ %
plt.subplot(1, 2, 2)
plt.pie(
    counts, 
    labels=splits, 
    autopct="%1.1f%%", 
    startangle=140, 
    colors=colors,
    explode=(0.05, 0),
    shadow=True,
    textprops={"fontsize": 11, "fontweight": "bold"}
)
plt.title("Dataset Split Ratio (%)", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()


### **Cell 9: Hiển thị mẫu thử nghiệm từ `train_dataset`**
- Lấy 1 mẫu ngẫu nhiên từ batch đầu tiên của `train_dataset` và vẽ đối chiếu giữa ảnh gốc và mask thực tế.


In [ ]:
def display(display_list):
    plt.figure(figsize=(10, 5))
    title = ["Input Image", "True Mask"]

    for i in range(len(display_list)):
        plt.subplot(1, len(display_list), i + 1)
        plt.title(title[i])
        plt.imshow(tf.keras.utils.array_to_img(display_list[i]))
        plt.axis("off")
    plt.show()

# Xem thử 1 mẫu từ train_dataset
for images, masks in train_dataset.take(1):
    sample_image, sample_mask = images[0], masks[0]
    display([sample_image, sample_mask])


### **Cell 10: Xây dựng kiến trúc mô hình U-Net (Backbone MobileNetV2)**
- **Encoder**: MobileNetV2 pretrained (đóng băng trọng số), trích xuất 5 mức feature map.
- **Decoder**: Các khối giải chập `Conv2DTranspose` + `BatchNorm` + `ReLU` kết nối Skip Connections.
- **Output**: Lớp `Conv2DTranspose` với `sigmoid` trả về mask 1 channel ($256 \times 256$).


In [ ]:
# 1. Base Model (Encoder / Downstack)
base_model = tf.keras.applications.MobileNetV2(
    input_shape=[IMG_HEIGHT, IMG_WIDTH, 3],
    include_top=False
)

# Các layer đầu ra dùng để nối Skip connections
layer_names = [
    "block_1_expand_relu",   # 128x128
    "block_3_expand_relu",   # 64x64
    "block_6_expand_relu",   # 32x32
    "block_13_expand_relu",  # 16x16
    "block_16_project",      # 8x8
]
base_model_outputs = [base_model.get_layer(name).output for name in layer_names]

# Đóng băng Encoder
down_stack = tf.keras.Model(inputs=base_model.input, outputs=base_model_outputs)
down_stack.trainable = False

# 2. Decoder Block (Upsampling)
def upsample(filters, size):
    initializer = tf.random_normal_initializer(0., 0.02)
    result = tf.keras.Sequential([
        tf.keras.layers.Conv2DTranspose(
            filters, size, strides=2, padding="same",
            kernel_initializer=initializer, use_bias=False
        ),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.ReLU()
    ])
    return result

up_stack = [
    upsample(512, 3),  # 8x8 -> 16x16
    upsample(256, 3),  # 16x16 -> 32x32
    upsample(128, 3),  # 32x32 -> 64x64
    upsample(64, 3),   # 64x64 -> 128x128
]

# 3. Kết hợp thành U-Net hoàn chỉnh
def unet_model(output_channels: int):
    inputs = tf.keras.layers.Input(shape=[IMG_HEIGHT, IMG_WIDTH, 3])

    # Trích xuất đặc trưng (Downsampling)
    skips = down_stack(inputs)
    x = skips[-1]
    skips = reversed(skips[:-1])

    # Giải chập & Nối Skip Connections (Upsampling)
    for up, skip in zip(up_stack, skips):
        x = up(x)
        concat = tf.keras.layers.Concatenate()
        x = concat([x, skip])

    # Output layer dự đoán mask nhị phân
    last = tf.keras.layers.Conv2DTranspose(
        filters=output_channels, kernel_size=3, strides=2,
        padding="same", activation="sigmoid"
    ) # 128x128 -> 256x256

    x = last(x)
    return tf.keras.Model(inputs=inputs, outputs=x)

# Khởi tạo mô hình
model = unet_model(output_channels=1)
model.summary()


### **Cell 11: Thiết lập Custom Loss (`BCE + Dice`), Metrics & Bộ Callbacks**
- Định nghĩa và đăng ký hàm Loss: `combined_loss = BCE + 0.8 * Dice_Loss`.
- Compile với optimizer `Adam(lr=1e-4)` và theo dõi chỉ số `BinaryIoU`.
- Thiết lập Callbacks: `ModelCheckpoint` (lưu best model), `ReduceLROnPlateau` (giảm LR), `EarlyStopping` (chống overfit).


In [ ]:
# 1. Định nghĩa Dice Loss và Combined Loss (BCE + Dice Loss)
@tf.keras.utils.register_keras_serializable(package="custom_losses")
def dice_loss(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(tf.cast(y_true, tf.float32), [-1])
    y_pred_f = tf.reshape(tf.cast(y_pred, tf.float32), [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    score = (2. * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)
    return 1. - score

@tf.keras.utils.register_keras_serializable(package="custom_losses")
def combined_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    dice = dice_loss(y_true, y_pred)
    return bce + 0.8 * dice

# 2. Compile mô hình với Combined Loss & BinaryIoU
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=combined_loss,
    metrics=["accuracy", tf.keras.metrics.BinaryIoU(target_class_ids=[1], name="iou")]
)

# 3. Hàm hiển thị kết quả dự đoán so với nhãn thật
def create_mask(pred_mask):
    pred_mask = tf.cast(pred_mask > 0.5, tf.float32)
    return pred_mask[0]

def show_predictions(dataset=None, num=1):
    if dataset:
        for image, mask in dataset.take(num):
            pred_mask = model.predict(image, verbose=0)
            display_list = [image[0], mask[0], create_mask(pred_mask)]
            
            plt.figure(figsize=(12, 4))
            titles = ["Input Image", "True Mask", "Predicted Mask"]
            for i in range(len(display_list)):
                plt.subplot(1, len(display_list), i + 1)
                plt.title(titles[i])
                plt.imshow(tf.keras.utils.array_to_img(display_list[i]))
                plt.axis("off")
            plt.show()

# 4. Bộ Callbacks nâng cao
class DisplayCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"\n--- Kết quả dự đoán sau Epoch {epoch+1} ---")
            show_predictions(val_dataset, 1)

os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)

callbacks = [
    DisplayCallback(),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=MODEL_SAVE_PATH,
        monitor="val_iou",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_iou",
        mode="max",
        patience=8,
        restore_best_weights=True,
        verbose=1
    )
]


### **Cell 12: Huấn luyện mô hình (`model.fit`)**
- Tiến hành huấn luyện qua tối đa 25 Epochs kết hợp tự động lưu best model và dừng sớm.


In [ ]:
# Tiến hành huấn luyện
EPOCHS = 25

history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=val_dataset,
    callbacks=callbacks
)


### **Cell 13: Vẽ đồ thị đánh giá hiệu năng (Loss, IoU, Accuracy)**
- Vẽ đường cong đánh giá sự hội tụ qua các Epoch giữa tập Train và Validation.


In [ ]:
# 1. Biểu đồ Loss
loss = history.history["loss"]
val_loss = history.history["val_loss"]
epochs_range = range(1, len(loss) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs_range, loss, label="Training Loss", color="royalblue", linewidth=2.5)
plt.plot(epochs_range, val_loss, label="Validation Loss", color="crimson", linestyle="--", linewidth=2.5)
plt.title("Loss Curve (Training vs Validation)", fontsize=14, fontweight="bold")
plt.xlabel("Epochs", fontsize=12)
plt.ylabel("Loss", fontsize=12)
plt.legend(loc="upper right", fontsize=11)
plt.grid(True, linestyle=":", alpha=0.6)
plt.show()

# 2. Biểu đồ IoU Score
iou = history.history.get("iou", history.history.get("binary_io_u", []))
val_iou = history.history.get("val_iou", history.history.get("val_binary_io_u", []))

plt.figure(figsize=(8, 5))
if len(iou) > 0:
    plt.plot(epochs_range, iou, label="Training IoU", color="royalblue", linewidth=2.5)
    plt.plot(epochs_range, val_iou, label="Validation IoU", color="forestgreen", linestyle="--", linewidth=2.5)
plt.title("IoU Score (Training vs Validation)", fontsize=14, fontweight="bold")
plt.xlabel("Epochs", fontsize=12)
plt.ylabel("IoU", fontsize=12)
plt.legend(loc="lower right", fontsize=11)
plt.grid(True, linestyle=":", alpha=0.6)
plt.show()

# 3. Biểu đồ Pixel Accuracy
acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]

plt.figure(figsize=(8, 5))
plt.plot(epochs_range, acc, label="Training Accuracy", color="royalblue", linewidth=2.5)
plt.plot(epochs_range, val_acc, label="Validation Accuracy", color="darkorange", linestyle="--", linewidth=2.5)
plt.title("Pixel Accuracy (Training vs Validation)", fontsize=14, fontweight="bold")
plt.xlabel("Epochs", fontsize=12)
plt.ylabel("Accuracy", fontsize=12)
plt.legend(loc="lower right", fontsize=11)
plt.grid(True, linestyle=":", alpha=0.6)
plt.show()


### **Cell 14: Hàm Inference Tách nền & Nắn thẳng hóa đơn (Perspective Warp)**
- Dự đoán mask trên ảnh kích thước gốc.
- Xóa nền đen (`image * mask`).
- Tìm 4 góc đỉnh và thực hiện **Perspective Transform** nắn phẳng hóa đơn.


In [ ]:
def order_points(pts):
    """Sắp xếp 4 điểm: top-left, top-right, bottom-right, bottom-left."""
    rect = np.zeros((4, 2), dtype="float32")
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]

    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]
    return rect

def four_point_transform(image, pts):
    """Nắn thẳng và crop ảnh hóa đơn theo 4 góc."""
    rect = order_points(pts)
    (tl, tr, br, bl) = rect

    widthA = np.sqrt(((br[0] - bl[0]) ** 2) + ((br[1] - bl[1]) ** 2))
    widthB = np.sqrt(((tr[0] - tl[0]) ** 2) + ((tr[1] - tl[1]) ** 2))
    maxWidth = max(int(widthA), int(widthB))

    heightA = np.sqrt(((tr[0] - br[0]) ** 2) + ((tr[1] - br[1]) ** 2))
    heightB = np.sqrt(((tl[0] - bl[0]) ** 2) + ((tl[1] - bl[1]) ** 2))
    maxHeight = max(int(heightA), int(heightB))

    dst = np.array([
        [0, 0],
        [maxWidth - 1, 0],
        [maxWidth - 1, maxHeight - 1],
        [0, maxHeight - 1]
    ], dtype="float32")

    M = cv2.getPerspectiveTransform(rect, dst)
    warped = cv2.warpPerspective(image, M, (maxWidth, maxHeight))
    return warped

def remove_background_from_path(image_path, apply_warp=True):
    """Dự đoán mask, tách nền và nắn thẳng hóa đơn."""
    orig_bgr = cv2.imread(image_path)
    if orig_bgr is None:
        print(f"Không tìm thấy hoặc không thể đọc ảnh: {image_path}")
        return None
    orig_rgb = cv2.cvtColor(orig_bgr, cv2.COLOR_BGR2RGB)
    h_orig, w_orig = orig_rgb.shape[:2]

    # Tiền xử lý cho AI Model
    img_resized = cv2.resize(orig_rgb, (IMG_WIDTH, IMG_HEIGHT))
    img_tensor = tf.cast(img_resized, tf.float32) / 255.0

    # Dự đoán mask
    pred = model.predict(tf.expand_dims(img_tensor, 0), verbose=0)
    pred_m = create_mask(pred).numpy()

    # Resize mask về kích thước gốc & tách nền
    mask_full = cv2.resize((pred_m > 0.5).astype(np.uint8), (w_orig, h_orig), interpolation=cv2.INTER_NEAREST)
    result_masked = orig_rgb * np.expand_dims(mask_full, axis=-1)

    # Tìm 4 góc & Perspective Warp
    warped_img = None
    if apply_warp:
        contours, _ = cv2.findContours((mask_full * 255).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if contours:
            c = max(contours, key=cv2.contourArea)
            peri = cv2.arcLength(c, True)
            approx = cv2.approxPolyDP(c, 0.02 * peri, True)
            
            if len(approx) == 4:
                pts = approx.reshape(4, 2).astype("float32")
                warped_img = four_point_transform(orig_rgb, pts)
            else:
                rect = cv2.minAreaRect(c)
                box = cv2.boxPoints(rect)
                warped_img = four_point_transform(orig_rgb, box.astype("float32"))

    # Hiển thị kết quả trực quan
    num_cols = 4 if warped_img is not None else 3
    plt.figure(figsize=(5 * num_cols, 5))

    plt.subplot(1, num_cols, 1)
    plt.title("1. Ảnh gốc", fontsize=12, fontweight="bold")
    plt.imshow(orig_rgb)
    plt.axis("off")

    plt.subplot(1, num_cols, 2)
    plt.title("2. Mask AI dự đoán", fontsize=12, fontweight="bold")
    plt.imshow(mask_full, cmap="gray")
    plt.axis("off")

    plt.subplot(1, num_cols, 3)
    plt.title("3. Đã loại bỏ Background", fontsize=12, fontweight="bold")
    plt.imshow(result_masked)
    plt.axis("off")

    if warped_img is not None:
        plt.subplot(1, num_cols, 4)
        plt.title("4. Nắn thẳng (Perspective Warp)", fontsize=12, fontweight="bold")
        plt.imshow(warped_img)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

    return {
        "masked": result_masked,
        "mask": mask_full,
        "warped": warped_img if warped_img is not None else result_masked
    }


In [ ]:
import time
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# 1. Đường dẫn model đã lưu
MODEL_PATH = r"e:\HOCKY7\PBL6\code_final\ai-service\models\1_segmentation\receipt_unet_mobilenetv2.keras"
IMG_HEIGHT = 256
IMG_WIDTH = 256

# 2. Load model (compile=False để load nhanh và tối ưu bộ nhớ khi chỉ inference)
print("Đang nạp model vào bộ nhớ...")
start_load = time.perf_counter()
model = tf.keras.models.load_model(MODEL_PATH, compile=False)

# Warmup model (chạy thử 1 tensor giả lập để khởi tạo tính toán trước)
_ = model(tf.zeros((1, IMG_HEIGHT, IMG_WIDTH, 3)), training=False)
print(f"-> Nạp model hoàn tất ({time.perf_counter() - start_load:.2f}s)!\n")

# 3. Các hàm xử lý hình học & nắn thẳng hóa đơn
def order_points(pts):
    rect = np.zeros((4, 2), dtype="float32")
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]
    return rect

def four_point_transform(image, pts):
    rect = order_points(pts)
    (tl, tr, br, bl) = rect
    widthA = np.sqrt(((br[0] - bl[0]) ** 2) + ((br[1] - bl[1]) ** 2))
    widthB = np.sqrt(((tr[0] - tl[0]) ** 2) + ((tr[1] - tl[1]) ** 2))
    maxWidth = max(int(widthA), int(widthB))

    heightA = np.sqrt(((tr[0] - br[0]) ** 2) + ((tr[1] - br[1]) ** 2))
    heightB = np.sqrt(((tl[0] - bl[0]) ** 2) + ((tl[1] - bl[1]) ** 2))
    maxHeight = max(int(heightA), int(heightB))

    dst = np.array([
        [0, 0],
        [maxWidth - 1, 0],
        [maxWidth - 1, maxHeight - 1],
        [0, maxHeight - 1]
    ], dtype="float32")

    M = cv2.getPerspectiveTransform(rect, dst)
    return cv2.warpPerspective(image, M, (maxWidth, maxHeight))

def process_receipt(image_path, show_plot=True):
    t_start = time.perf_counter()

    # --- Bước 1: Tiền xử lý (Preprocessing) ---
    t0 = time.perf_counter()
    orig_bgr = cv2.imread(image_path)
    if orig_bgr is None:
        raise FileNotFoundError(f"Không thể đọc ảnh từ: {image_path}")
    orig_rgb = cv2.cvtColor(orig_bgr, cv2.COLOR_BGR2RGB)
    h_orig, w_orig = orig_rgb.shape[:2]

    img_resized = cv2.resize(orig_rgb, (IMG_WIDTH, IMG_HEIGHT))
    img_tensor = tf.cast(img_resized, tf.float32) / 255.0
    img_input = tf.expand_dims(img_tensor, axis=0)
    t_pre = (time.perf_counter() - t0) * 1000  # ms

    # --- Bước 2: AI Dự đoán (Model Inference) ---
    t1 = time.perf_counter()
    pred = model(img_input, training=False)
    pred_mask = (pred[0, :, :, 0].numpy() > 0.5).astype(np.uint8)
    t_infer = (time.perf_counter() - t1) * 1000  # ms

    # --- Bước 3: Hậu xử lý & Nắn thẳng (Postprocessing & Warp) ---
    t2 = time.perf_counter()
    mask_full = cv2.resize(pred_mask, (w_orig, h_orig), interpolation=cv2.INTER_NEAREST)
    result_masked = orig_rgb * np.expand_dims(mask_full, axis=-1)

    contours, _ = cv2.findContours(mask_full * 255, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    warped_img = None
    if contours:
        c = max(contours, key=cv2.contourArea)
        peri = cv2.arcLength(c, True)
        approx = cv2.approxPolyDP(c, 0.02 * peri, True)
        if len(approx) == 4:
            pts = approx.reshape(4, 2).astype("float32")
        else:
            rect = cv2.minAreaRect(c)
            pts = cv2.boxPoints(rect).astype("float32")
        warped_img = four_point_transform(orig_rgb, pts)

    t_post = (time.perf_counter() - t2) * 1000  # ms
    t_total = (time.perf_counter() - t_start) * 1000  # ms

    # In chi tiết thời gian xử lý
    print("=" * 45)
    print(f"⏱️ THỜI GIAN XỬ LÝ (Kích thước gốc: {w_orig}x{h_orig}px):")
    print(f" • Preprocessing      : {t_pre:6.2f} ms")
    print(f" • Model Inference    : {t_infer:6.2f} ms (chỉ phần AI)")
    print(f" • Postprocessing/Warp: {t_post:6.2f} ms")
    print(f"---------------------------------------------")
    print(f" 🚀 TỔNG THỜI GIAN     : {t_total:6.2f} ms (~{1000/t_total:.1f} FPS)")
    print("=" * 45)

    if show_plot:
        cols = 4 if warped_img is not None else 3
        plt.figure(figsize=(4.5 * cols, 4))
        
        plt.subplot(1, cols, 1)
        plt.title("1. Ảnh gốc", fontweight="bold")
        plt.imshow(orig_rgb)
        plt.axis("off")

        plt.subplot(1, cols, 2)
        plt.title("2. Mask AI", fontweight="bold")
        plt.imshow(mask_full, cmap="gray")
        plt.axis("off")

        plt.subplot(1, cols, 3)
        plt.title("3. Tách nền", fontweight="bold")
        plt.imshow(result_masked)
        plt.axis("off")

        if warped_img is not None:
            plt.subplot(1, cols, 4)
            plt.title("4. Nắn phẳng", fontweight="bold")
            plt.imshow(warped_img)
            plt.axis("off")

        plt.tight_layout()
        plt.show()

    return warped_img if warped_img is not None else result_masked


In [ ]:
# 👉 Đổi đường dẫn tới ảnh bạn muốn kiểm tra ở đây:
IMAGE_PATH = r"E:\HOCKY7\PBL6\code_final\ai-service\data\1_segmentation\imgs\mcocr_public_145014zrswr.jpg"

# Chạy và xem tốc độ xử lý
warped_result = process_receipt(IMAGE_PATH, show_plot=True)
